# Initial Sync Test

# Notes

- Uses the Python Client: Furhat Realtime API, v0.1.3, 27/10/25: https://pypi.org/project/furhat-realtime-api/
- NOT the Python Client: Furhat Remote API, v1.0.2, 15/11/21: https://pypi.org/project/furhat-remote-api/#description

## Virtual Furhat Setup

- SDK for: i) Launcher to run the Virtual Furhat; ii) Virtual Furhat is the simulation.
- Follow instructions to create SDK account, dowlnload launcher and get API key: https://docs.furhat.io/setup/sdk
- Run the Launcher for the Furhat Studio: i) Virtual Furhat should run; ii) Web interface for custom control

- Websocket is: ws://<ROBOT_IP>:9000/v1/events eg ws://127.0.0.1:9000/v1/events
- Websocket playground using: http://127.0.0.1:9000/v1/index.html

## Furhat Realtime API

- Realtime API Intro: https://docs.furhat.io/realtime-api/intro
- Python client package (and API reference): https://pypi.org/project/furhat-realtime-api/
- API documentation: i) PyPi package; ii) Websocket playground; iii) API examples, https://github.com/FurhatRobotics/realtime-api-examples/tree/main/python

## Artifical Social Agent (ASA) Application

- A UV managed and pacjaged Python application
- TBC: setup instructions


In [1]:
# Launch with all debugs

import argparse
import logging

from asa import __version__
from asa._tools import depreport, portcheck
from asa._tools.custom_logging import setup_logging

log = logging.getLogger(f"{__name__}.app")

# Dummy CLI args (mirrors asa.cli's argparse.Namespace)
args = argparse.Namespace(log="debug", host="127.0.0.1")

# Establish custom logging
setup_logging(level=args.log.upper())

# Launch and versions
print("Artificial Social Agent")
print(f"ASA Version: {__version__}")
depreport.report()

# Quick test of logging
# log.info("ASA Launch")
log.debug("Log test Debug")
log.info("Log test Info")
log.warning("Log test Warning")

# Double-check kernels and ports
print("Ports Check")
portcheck.report()

Artificial Social Agent
ASA Version: 0.3.0.dev0
PACKAGE              INSTALLED  RELEASED    GROUP     DECLARED
-------------------  ---------  ----------  --------  --------------------------
furhat-realtime-api  0.1.3      2025-10-27  project   furhat-realtime-api>=0.1.3
ipykernel            7.3.0      2026-06-10  notebook  ipykernel>=7.3.0
ipywidgets           8.1.8      2025-11-01  notebook  ipywidgets>=8.1.8
jupyterlab           4.6.2      2026-07-21  notebook  jupyterlab>=4.6.1
nbclient             0.11.0     2026-06-05  notebook  nbclient>=0.11.0
nbformat             5.10.4     2024-04-04  notebook  nbformat>=5.10.4
ruff                 0.16.0     2026-07-23  dev       ruff>=0.15.20
autopep8             2.3.2      2025-01-14  dev       autopep8>=2.3.2
pytest               9.1.1      2026-06-19  dev       pytest>=9.1.1
DEBUG: __main__.app.<module>.line_25 - Log test Debug
INFO: __main__.app.<module>.line_26 - Log test Info
Ports Check
PID    PROJECT                      AGE    POR

[Kernel(pid=15149, project='Repo asa_research_prototype', ports=(9005, 9006, 9007, 9008, 9009), age='00:05', connection_file=PosixPath('/Users/stuartgow/Library/Jupyter/runtime/kernel-v320a60ba2b18b25cfc271ed02981bfbc3daf765fc.json'), is_current=True)]

In [ ]:
# from asa._tools import portcheck

# portcheck.report()               # what's alive, what's litter, who holds port 9000
# portcheck.clean()                # dry run — counts what would go, deletes nothing
# portcheck.clean(dry_run=False)   # actually delete

---

# Establish a Furhat Session

In [2]:
from asa import ASASession

host = args.host
client_log_level = getattr(logging, args.log.upper())

session = ASASession(host=host, client_log_level=client_log_level)
await session.start()   # keep it open across cells

INFO: asa.session.app.start.line_60 - Session starting - host 127.0.0.1


FurhatUnreachable: No Furhat at ws://127.0.0.1:9000/v1/events — is the Furhat launcher running? ([Errno 61] Connect call failed ('127.0.0.1', 9000))

In [3]:
await session.gesture("BigSmile", intensity=0.9, duration=4.0)
await session.say("Hello, this is a 99th test")

RuntimeError: Session not started — call start() first

In [4]:
await session.stop()

---

In [ ]:
# await test_furhat.request_gesture_start(name="Smile", intensity=0.6, duration=2.0)

# await asyncio.gather(
#     test_furhat.request_speak_text("Lovely to see you", wait=True, abort=True),
#     test_furhat.request_gesture_start(name="BigSmile", intensity=0.9, duration=4.0, wait=True),
#     test_furhat.request_gesture_start(name="Blink", intensity=0.8, duration=6.0, wait=True),
# )

## Future use

This is why the async client is the right one for your prototype:

- asyncio.gather(...) — speak and gesture and set the LED as one turn, rather than three round-trips end to end.
- asyncio.wait_for(coro, timeout) — the client already uses it internally with a 5s default. A user who says nothing shouldn't hang your agent.
- task.cancel() — barge-in. User starts talking while Furhat is mid-sentence, you cancel the speaking task. The abort=True flag you're passing is the protocol-level version of the same idea.

Handling perception while acting — the listener task means you can react to a response_hear event that arrives during an utterance. Impossible if your thread is blocked inside a speak call.

In [ ]:
import asyncio
import time


async def job(name, seconds):
    print(f"{name} start   {time.strftime('%H:%M:%S')}")
    await asyncio.sleep(seconds)
    print(f"{name} done    {time.strftime('%H:%M:%S')}")
    return name

results = await asyncio.gather(job("A", 3), job("B", 1), job("C", 2))
print(results, "— elapsed ~3s, not 6")